# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayesha-asif1/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I'm predicting CTR (a continuous ratio between 0 and 1), so this is a regression problem, not classification. In Week 4's leakage trap, honest Linear Regression scored R²=0.0018 and honest Decision Tree scored R²=-0.0214 — both weak, and the Decision Tree overfit on its own. I'm choosing Random Forest Regressor because it averages many trees, which reduces the overfitting a single Decision Tree showed, and it can capture non-linear relationships (e.g., CTR likely doesn't fall linearly as position gets worse) that Linear Regression couldn't. I'll also use permutation importance to interpret which features it actually leans on.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

My data has repeated content_hash_id values across report dates within March 2026. A random row-level split risks leaking the same content into both train and test, inflating the score artificially. So I'm using a grouped split by content_hash_id — all rows for a given piece of content stay entirely in either train or test, never both. This mirrors how the model would actually be used: predicting CTR for content it has never seen scored before.

In [2]:
import pandas as pd
from huggingface_hub import login
from google.colab import userdata

# authenticate with your gated HF token (same as ML-04)
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# load the March 2026 partition
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/",
)

print(f"Rows loaded: {len(df_march)}")
print(df_march.columns.tolist())
df_march['report_date'] = pd.to_datetime(df_march['report_date'])
df_march['day_of_week'] = df_march['report_date'].dt.day_name()

print(df_march[['report_date', 'day_of_week']].head())

Rows loaded: 9841378
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
  report_date day_of_week
0  2026-03-01      Sunday
1  2026-03-01      Sunday
2  2026-03-01      Sunday
3  2026-03-01      Sunday
4  2026-03-01      Sunday


In [3]:
# filter to usable rows
df_march = df_march[df_march['gsc_data_available'] == True].copy()
print(f"Usable rows: {len(df_march)}")

# rebuild the label
df_march['ctr'] = df_march['gsc_clicks'] / df_march['gsc_impressions']

# rebuild features from ML-04
import numpy as np
df_march['log_impressions'] = np.log1p(df_march['gsc_impressions'])

def bucket_position(pos):
    if pos <= 3:
        return 'top_3'
    elif pos <= 10:
        return 'top_10'
    elif pos <= 20:
        return 'top_20'
    else:
        return 'beyond_20'

df_march['position_bucket'] = df_march['gsc_avg_position'].apply(bucket_position)

print(df_march[['ctr', 'log_impressions', 'position_bucket']].head())

Usable rows: 3611061
     ctr  log_impressions position_bucket
0  0.000         3.044522          top_10
1  0.000         0.693147           top_3
2  0.008         4.836282          top_10
3  0.000         2.079442          top_10
4  0.000         2.484907           top_3


In [4]:
from sklearn.model_selection import GroupShuffleSplit

features = ['gsc_impressions', 'log_impressions', 'gsc_avg_position', 'day_of_week', 'position_bucket']

X = df_march[features].copy()
y = df_march['ctr']
groups = df_march['content_hash_id']

# one-hot encode categorical columns
X = pd.get_dummies(X, columns=['position_bucket', 'day_of_week'], drop_first=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Unique content in train: {df_march.iloc[train_idx]['content_hash_id'].nunique()}")
print(f"Unique content in test: {df_march.iloc[test_idx]['content_hash_id'].nunique()}")
overlap = set(df_march.iloc[train_idx]['content_hash_id']) & set(df_march.iloc[test_idx]['content_hash_id'])
print(f"Overlapping content_hash_ids: {len(overlap)}")  # should be 0

Train rows: 2888030, Test rows: 723031
Unique content in train: 141390
Unique content in test: 35348
Overlapping content_hash_ids: 0


## 3. Train + compare vs my baseline

I compared Random Forest against a mean-CTR baseline on the same grouped test split. Random Forest achieved R²=0.0051 versus the baseline's R²≈0, and a marginally lower MAE (0.005436 vs 0.005485). The improvement is real but small — the model explains only about 0.5% of CTR variance. This confirms CTR is dominated by factors outside these five features (actual content quality, title/meta text, competing results) rather than by position, volume, or day of week alone.

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Baseline: predict mean CTR from training set (no-signal baseline)
baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

# Random Forest
rf = RandomForestRegressor(n_estimators=50, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

results = pd.DataFrame({
    'model': ['Mean baseline', 'Random Forest'],
    'R2': [r2_score(y_test, baseline_pred), r2_score(y_test, rf_pred)],
    'MAE': [mean_absolute_error(y_test, baseline_pred), mean_absolute_error(y_test, rf_pred)]
})
print(results)

           model            R2       MAE
0  Mean baseline -8.252001e-08  0.005485
1  Random Forest  5.086436e-03  0.005436


## 4. Errors and interpretation

gsc_avg_position is by far the strongest signal (importance 0.0114), followed by log_impressions. day_of_week and position_bucket contribute almost nothing once raw position is available — likely because position_bucket was derived from gsc_avg_position, so it adds no new information. The largest individual errors occur where actual CTR was 1.0 (a single click on a single impression) — a statistical fluke the model cannot learn to predict. Directionally, mean absolute error is lowest in beyond_20 (0.0024) and highest in top_3 (0.0085): low-ranked content has consistently near-zero CTR and is easy to predict, while top-ranked content has much higher CTR variance that these features don't explain.

In [6]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)
print(importance_df)

# error analysis
errors = pd.DataFrame({
    'actual': y_test.values,
    'predicted': rf_pred,
    'abs_error': np.abs(y_test.values - rf_pred)
})
print(errors.sort_values('abs_error', ascending=False).head(10))
print("\nMean abs error by position bucket:")
errors_with_bucket = errors.copy()
errors_with_bucket['position_bucket'] = df_march.iloc[test_idx]['position_bucket'].values
print(errors_with_bucket.groupby('position_bucket')['abs_error'].mean())

                   feature    importance
2         gsc_avg_position  1.137179e-02
1          log_impressions  3.334173e-03
0          gsc_impressions  2.175793e-03
7     day_of_week_Saturday  9.734731e-05
8       day_of_week_Sunday  8.779691e-05
10     day_of_week_Tuesday  2.708111e-05
6       day_of_week_Monday  4.078127e-07
4   position_bucket_top_20  2.731998e-07
3   position_bucket_top_10 -2.499033e-06
5    position_bucket_top_3 -2.588513e-06
11   day_of_week_Wednesday -3.579062e-06
9     day_of_week_Thursday -1.207075e-04
        actual  predicted  abs_error
684017     1.0   0.000025   0.999975
370976     1.0   0.000032   0.999968
277555     1.0   0.000037   0.999963
434921     1.0   0.000257   0.999743
520859     1.0   0.001223   0.998777
571185     1.0   0.001498   0.998502
440514     1.0   0.001509   0.998491
394304     1.0   0.001604   0.998396
705869     1.0   0.001718   0.998282
45061      1.0   0.001740   0.998260

Mean abs error by position bucket:
position_bucket
beyond_2

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# --- "Before" — naive random row-level split (no grouping) ---
X_naive_train, X_naive_test, y_naive_train, y_naive_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_naive = RandomForestRegressor(n_estimators=50, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_naive.fit(X_naive_train, y_naive_train)
naive_pred = rf_naive.predict(X_naive_test)

print("=== BEFORE: naive random split (content leakage across train/test) ===")
print(f"R²: {r2_score(y_naive_test, naive_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_naive_test, naive_pred):.6f}")

print("\n=== AFTER: honest grouped split (from Week 5) ===")
print(f"R²: {r2_score(y_test, rf_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, rf_pred):.6f}")

=== BEFORE: naive random split (content leakage across train/test) ===
R²: 0.0062
MAE: 0.005415

=== AFTER: honest grouped split (from Week 5) ===
R²: 0.0051
MAE: 0.005436


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.